### LLM : 질의응답 챗봇 개발 파이프라인

- 이 노트북은 **`질의 응답 챗봇` 개발 파이프라인 예제 실습**을 수행하는 **Google Colab용** 노트북입니다.

##### Llama를 활용한 도메인 특화 챗봇 개발 파이프라인 예제
   1.  Llama를 활용한 질의응답 챗봇 Fine-Tuning 및 성능 평가 (Cell 6개)

### Llama를 활용한 질의응답 챗봇 Fine-Tuning 및 성능 평가
사전학습된 Llama 3.2 Korean Bllossom 모델을 KoAlpaca 데이터셋으로 Fine-tuning하여 한국어 질의응답 성능을 향상시킵니다.

LoRA(Low-Rank Adaptation) 기법을 활용하여 효율적인 파라미터 튜닝을 수행하고 ROUGE 및 F1 Score로 성능을 정량적으로 평가합니다.

* KoAlpaca 데이터셋 로드 및 한국어 텍스트 전처리로 instruction-input-output 구조를 질의응답 형태로 변환
* Bllossom Llama 3.2 3B 모델을 4-bit 양자화로 로드하고 메모리 효율성 확보
* 대화형 메시지 포맷으로 데이터 변환하여 시스템-사용자-어시스턴트 역할 구조화
* LoRA 설정으로 어텐션 모듈만 선택적 학습하여 계산 비용 절감 및 과적합 방지
* SFTTrainer를 사용한 지도학습 Fine-tuning 실행 및 학습 과정 모니터링
* ROUGE-1/2/L 및 F1 Score 기반 정량적 성능 평가와 샘플 예측 결과 확인

In [1]:
!pip install -q transformers==4.52.4
!pip install -q peft==0.12.0
!pip install -q trl==0.18.2
!pip install -q datasets
!pip install -q bitsandbytes==0.46.0
!pip install -q accelerate
!pip install -q rouge-score
!pip install -q sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.4/366.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 14.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# ============================================
# Cell 1: 환경 설정 및 라이브러리 임포트
# ============================================

import torch
import transformers
import peft
import os
import warnings
warnings.filterwarnings('ignore')  # 경고 메시지 숨김

# 현재 환경 정보 출력
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
# GPU가 사용 가능한 경우에만 GPU 정보 출력
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print(f"GPU 이름: N/A")
    print(f"GPU 메모리: N/A")

print(f"Transformers 버전: {transformers.__version__}")
print(f"PEFT 버전: {peft.__version__}")

# GPU 연산 속도 최적화 설정 (GPU 환경에서만 적용)
if torch.cuda.is_available():
    os.environ["CUDA_LAUNCH_BLOCKING"] = "0"  # CUDA 커널 비동기 실행
    torch.backends.cuda.matmul.allow_tf32 = True    # TensorFloat-32 연산 활성화
    torch.backends.cudnn.allow_tf32 = True          # cuDNN TF32 연산 활성화
    torch.backends.cudnn.benchmark = True           # 최적 알고리즘 자동 선택

PyTorch 버전: 2.9.0+cu126
CUDA 사용 가능: True
GPU 이름: Tesla T4
GPU 메모리: 14.74 GB
Transformers 버전: 4.52.4
PEFT 버전: 0.12.0


In [3]:
# ============================================
# Cell 2: 데이터셋 로드 및 전처리
# ============================================

from datasets import load_dataset
import re
from typing import Dict

# KoAlpaca 한국어 instruction 데이터셋 로드
dataset = load_dataset("Taegyuu/KoAlpaca-v1.1a", trust_remote_code=True)

# 데이터셋 기본 정보 확인
print(f"데이터셋 로드 완료!")
print(f"학습 데이터: {len(dataset['train'])} 샘플")
print(f"\n데이터셋 구조:")
print(dataset)

# 첫 번째 샘플의 구조 분석
print("\n샘플 데이터 확인:")
sample = dataset['train'][0]
print(sample)
# 각 필드의 내용 미리보기 (100자 제한)
for key in sample.keys():
    print(f"{key}: {str(sample[key])[:100]}...")

def preprocess_korean_text(text: str) -> str:
    """한국어 텍스트 정리 및 정규화"""
    # HTML/JavaScript 코드 완전 제거
    text = re.sub(r'<script.*?</script>', '', text, flags=re.DOTALL)
    text = re.sub(r'<style.*?</style>', '', text, flags=re.DOTALL)

    # HTML 태그 제거
    text = re.sub(r'<[^>]+>', '', text)

    # HTML 특수문자 변환
    text = text.replace('&nbsp;', ' ')
    text = text.replace('&lt;', '<')
    text = text.replace('&gt;', '>')
    text = text.replace('&amp;', '&')
    text = text.replace('&quot;', '"')

    # 한글, 영문, 숫자, 기본 문장부호만 유지
    text = re.sub(r'[^\w\s가-힣.,!?0-9%]', ' ', text)

    # 연속된 공백을 하나로 통합
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def process_koalpaca_example(example: Dict) -> Dict:
    """KoAlpaca 형식을 표준 QA 형식으로 변환"""
    # instruction, input, output 필드 전처리
    instruction = preprocess_korean_text(example.get('instruction', ''))
    input_text = preprocess_korean_text(example.get('input', ''))
    output = preprocess_korean_text(example.get('output', ''))

    # instruction과 input을 결합하여 완전한 질문 생성
    if input_text.strip():
        question = f"{instruction}\n\n{input_text}"
    else:
        question = instruction

    # 표준 QA 형식으로 반환
    return {
        'id': f"koalpaca_{hash(instruction)}",  # 고유 ID 생성
        'context': '',  # KoAlpaca는 별도 context 없음
        'question': question,
        'answer': output
    }

# 전체 데이터셋 전처리 실행
print("데이터 전처리 중...")
processed_dataset = dataset.map(process_koalpaca_example)

# 학습 시간 단축을 위한 샘플 수 제한
max_train_samples = 1000  # 학습용 샘플 수
max_eval_samples = 100    # 검증용 샘플 수

if len(processed_dataset['train']) > max_train_samples:
    processed_dataset['train'] = processed_dataset['train'].select(range(max_train_samples))

# validation set이 없으므로 train에서 분할
print("Validation set이 없어서 train 데이터에서 분할합니다.")
total_samples = len(processed_dataset['train'])
train_end = total_samples - max_eval_samples

# 마지막 샘플들을 검증용으로 분리
processed_dataset['validation'] = processed_dataset['train'].select(range(train_end, total_samples))
processed_dataset['train'] = processed_dataset['train'].select(range(train_end))

print(f"전처리 완료!")
print(f"학습 데이터: {len(processed_dataset['train'])} 샘플")
print(f"검증 데이터: {len(processed_dataset['validation'])} 샘플")

# 전처리 결과 확인
print("\n전처리된 샘플:")
sample = processed_dataset['train'][0]
print(f"질문: {sample['question']}")
print(f"답변: {sample['answer']}")
print(f"문맥: {sample['context'][:100]}...")

print(f"\n데이터셋 컬럼: {processed_dataset['train'].column_names}")
print(f"데이터셋 특징: {processed_dataset['train'].features}")

def format_conversational_data(message):
    """표준 QA 형식을 Llama 대화 형식으로 변환"""
    question = message['question']
    context = message.get('context', None)
    answer = message.get('answer', None)

    # 시스템 프롬프트 정의
    system_message = "당신은 한국어 질의응답 전문가입니다. 주어진 문맥을 바탕으로 정확하고 간결한 답변을 제공하세요."

    # 문맥 유무에 따른 사용자 메시지 구성
    if context and context.strip():
        user_message = f"다음 문서를 참고하여 질문에 답변해주세요.\n\n[참고 문서]\n{context}\n\n[질문]\n{question}"
    else:
        user_message = question

    # 학습용: 답변 포함된 완전한 대화
    if answer:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer}
        ]
    # 추론용: 답변 없는 대화 (생성 대기)
    else:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": [{"type": "text", "text": user_message}]}
        ]

    return {"messages": messages}

# 학습용 대화 형식 변환
print("데이터셋을 conversational 형식으로 변환 중...")
train_dataset = processed_dataset['train'].map(format_conversational_data,
    remove_columns=processed_dataset['train'].column_names)
eval_dataset = processed_dataset['validation'].map(format_conversational_data,
    remove_columns=processed_dataset['validation'].column_names)

print("변환 완료!")
print(f"최종 학습 데이터: {len(train_dataset)} 샘플")
print(f"최종 검증 데이터: {len(eval_dataset)} 샘플")

# 최종 변환 결과 확인
print("\n=== 최종 샘플 확인 ===")
final_sample = train_dataset[0]
print("Messages:")
for i, msg in enumerate(final_sample['messages']):
    print(f"  {i+1}. {msg['role']}: {msg['content'][:150]}{'...' if len(msg['content']) > 150 else ''}")


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Taegyuu/KoAlpaca-v1.1a' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Taegyuu/KoAlpaca-v1.1a' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

KoAlpaca_v1.1.jsonl:   0%|          | 0.00/24.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21155 [00:00<?, ? examples/s]

데이터셋 로드 완료!
학습 데이터: 21155 샘플

데이터셋 구조:
DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'url'],
        num_rows: 21155
    })
})

샘플 데이터 확인:
{'instruction': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?', 'output': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.', 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268'}
instruction: 양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?...
output: 양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 

식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파...
url: https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268...
데이터 전처리 중...


Map:   0%|          | 0/21155 [00:00<?, ? examples/s]

Validation set이 없어서 train 데이터에서 분할합니다.
전처리 완료!
학습 데이터: 900 샘플
검증 데이터: 100 샘플

전처리된 샘플:
질문: 양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?
답변: 양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다. 덧붙이는 답변 고구마 줄기도 볶아먹을 수 있나요? 고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.
문맥: ...

데이터셋 컬럼: ['instruction', 'output', 'url', 'id', 'context', 'question', 'answer']
데이터셋 특징: {'instruction': Value('string'), 'output': Value('string'), 'url': Value('string'), 'id': Value('string'), 'context': Value('string'), 'question': Value('string'), 'answer': Value('string')}
데이터셋을 conversational 형식으로 변환 중...


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

변환 완료!
최종 학습 데이터: 900 샘플
최종 검증 데이터: 100 샘플

=== 최종 샘플 확인 ===
Messages:
  1. system: 당신은 한국어 질의응답 전문가입니다. 주어진 문맥을 바탕으로 정확하고 간결한 답변을 제공하세요.
  2. user: 양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?
  3. assistant: 양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니...


In [4]:
# ============================================
# Cell 3: 모델 및 토크나이저 로드
# ============================================

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 한국어 특화 Llama 모델 선택
model_name = "Bllossom/llama-3.2-Korean-Bllossom-3B"
print(f"선택된 모델: {model_name}")

# 메모리 절약을 위한 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 양자화 활성화
    bnb_4bit_quant_type="nf4",           # NormalFloat4 양자화 타입
    bnb_4bit_compute_dtype=torch.float16, # 연산용 데이터 타입
    bnb_4bit_use_double_quant=True,      # 이중 양자화로 메모리 추가 절약
)

# 모델 로드 (양자화 적용)
print("모델 로딩 중... (몇 분 소요될 수 있습니다)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,      # 양자화 설정 적용
    device_map="auto",                   # 자동 GPU 할당
    torch_dtype=torch.float16,           # 반정밀도 부동소수점
    trust_remote_code=True,              # 사용자 정의 코드 신뢰
    offload_folder="./offload"
)

# 토크나이저 로드 및 설정
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # 패딩 토큰 설정
tokenizer.padding_side = "right"               # 오른쪽 패딩

print("모델 및 토크나이저 로드 완료!")
print(f"모델 크기: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B 파라미터")

# 초기 모델 성능 테스트
test_context = "서울특별시는 대한민국의 수도이며, 인구는 약 950만 명입니다."
test_question = "한국의 수도는 어디인가요?"
template = format_conversational_data({'context': test_context, 'question': test_question})
test_prompt = tokenizer.apply_chat_template(
        template['messages'],
        tokenize=False,
        add_generation_prompt=True
    )

# 입력 토큰화 및 GPU 이동
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

print("초기 추론 테스트:")
print(f"질문: {test_question}")

# 테스트 추론 실행
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,                # 최대 생성 토큰 수
        temperature=0.7,                  # 생성 다양성 조절
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# 생성된 답변만 추출하여 출력
response = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print(f"답변: {response}")

선택된 모델: Bllossom/llama-3.2-Korean-Bllossom-3B
모델 로딩 중... (몇 분 소요될 수 있습니다)


config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

모델 및 토크나이저 로드 완료!
모델 크기: 1.80B 파라미터
초기 추론 테스트:
질문: 한국의 수도는 어디인가요?
답변: 서울입니다.


In [6]:
# ============================================
# Cell 4: LoRA Fine-tuning 설정 및 실행
# ============================================

from peft import LoraConfig
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

# LoRA(Low-Rank Adaptation) 파라미터 설정
lora_config = LoraConfig(
    r=32,                    # 저차원 행렬의 rank (클수록 표현력 증가)
    lora_alpha=64,           # 스케일링 팩터 (일반적으로 2*r)
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], # Llama 어텐션 모듈
    lora_dropout=0.05,       # 과적합 방지용 드롭아웃
    bias="none",             # 바이어스 학습 비활성화
    task_type="CAUSAL_LM"    # 인과적 언어모델 태스크
)

# Fine-tuning 하이퍼파라미터 설정
training_args = SFTConfig(
    output_dir="./korean-qa-lora",       # 모델 저장 경로
    num_train_epochs=1,                  # 학습 에포크 수
    per_device_train_batch_size=2,       # 훈련 배치 크기
    per_device_eval_batch_size=2,        # 평가 배치 크기
    gradient_checkpointing=True,         # 메모리 절약을 위한 체크포인팅
    gradient_accumulation_steps=4,       # 그래디언트 누적 단계
    optim="paged_adamw_8bit",           # 8-bit AdamW 옵티마이저
    logging_steps=10,                    # 로깅 주기
    logging_first_step=True,             # 첫 스텝 로깅
    logging_strategy="steps",            # 스텝 기반 로깅
    learning_rate=2e-4,                  # 학습률
    warmup_steps=100,                    # 워밍업 스텝
    save_strategy="steps",               # 모델 저장 전략
    save_steps=200,                      # 저장 주기
    eval_strategy="steps",               # 평가 전략
    eval_steps=50,                       # 평가 주기
    lr_scheduler_type="cosine",          # 코사인 학습률 스케줄러
    fp16=True,                           # 16-bit 부동소수점 연산
    max_seq_length=512,                  # 최대 시퀀스 길이
    packing=False,                       # 시퀀스 패킹 비활성화
    report_to="none",                    # 외부 로깅 도구 사용 안함
    max_grad_norm=1.0,                   # 그래디언트 클리핑
    seed=42,                             # 재현 가능성을 위한 시드
    dataloader_num_workers=4,            # 데이터 로더 워커 수
    dataloader_pin_memory=True,          # 메모리 핀 고정
    dataset_num_proc=4,                  # 데이터 전처리 병렬화
    neftune_noise_alpha=5,               # NEFTune 노이즈로 학습 안정성 향상
)

print("학습 설정 완료!")

# 데이터셋 샘플 최종 확인
print("학습 데이터셋 최종 샘플 출력")
print(train_dataset[0])
print(eval_dataset[0])

# SFTTrainer 초기화 (Supervised Fine-Tuning)
trainer = SFTTrainer(
    model=model,                         # 학습할 모델
    args=training_args,                  # 학습 인자
    train_dataset=train_dataset,         # 훈련 데이터
    eval_dataset=eval_dataset,           # 검증 데이터
    processing_class=tokenizer,          # 토크나이저 전달
    peft_config=lora_config,             # LoRA 설정
)

print("Fine-tuning 시작!")

# 실제 Fine-tuning 실행
trainer.train()

print("Fine-tuning 완료!")
# 학습 결과 요약 출력
print(trainer.state.log_history[-1])  # 마지막 로그
losses = [log['loss'] for log in trainer.state.log_history if 'loss' in log]
print(f"Loss 변화: {losses[0]} → {losses[-1]}")

# 학습된 모델 저장
trainer.save_model("./korean-qa-lora")
tokenizer.save_pretrained("./korean-qa-lora")

print("모델 저장 완료!")
print("저장 위치: ./korean-qa-lora")

학습 설정 완료!
학습 데이터셋 최종 샘플 출력
{'messages': [{'content': '당신은 한국어 질의응답 전문가입니다. 주어진 문맥을 바탕으로 정확하고 간결한 답변을 제공하세요.', 'role': 'system'}, {'content': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?', 'role': 'user'}, {'content': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다. 덧붙이는 답변 고구마 줄기도 볶아먹을 수 있나요? 고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.', 'role': 'assistant'}]}
{'messages': [{'content': '당신은 한국어 질의응답 전문가입니다. 주어진 문맥을 바탕으로 정확하고 간결한 답변을 제공하세요.', 'role': 'system'}, {'content': '일본의 일촌일품이란 정책은 무엇인가요?', 'role': 'user'}, {'content': '일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다. 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다. 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만, 대형 유원지나 교육시설을 개발하여 이를 주요 명소로 만드는 경우도 있습니다. 이에 따라 일촌일품의 주제는 상당히 폭넓고 다양합니다. 일본의 지리적 특성과 기후 차이를 이용하여 개발하는 경우가 많은데, 이

Converting train dataset to ChatML (num_proc=4):   0%|          | 0/900 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=4):   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=4):   0%|          | 0/900 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=4):   0%|          | 0/900 [00:00<?, ? examples/s]

...

Truncating eval dataset (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Fine-tuning 시작!


Step,Training Loss,Validation Loss
50,2.050200,2.047661
100,1.947300,1.961366


Fine-tuning 완료!
{'train_runtime': 717.2306, 'train_samples_per_second': 1.255, 'train_steps_per_second': 0.158, 'total_flos': 5747664865468416.0, 'train_loss': 2.1339532527248415, 'num_tokens': 288187.0, 'mean_token_accuracy': 0.6278310120105743, 'epoch': 1.0, 'step': 113}
Loss 변화: 2.9532 → 1.9169
모델 저장 완료!
저장 위치: ./korean-qa-lora


In [7]:
# ============================================
# Cell 5: 답변 생성 함수 정의
# ============================================

def generate_answer(question: str, context: str, max_length: int = 100) -> str:
    """Fine-tuning된 모델로 질문에 대한 답변 생성"""
    # 입력을 대화 형식으로 변환
    template = format_conversational_data({'context': context, 'question': question})
    prompt = tokenizer.apply_chat_template(
        template['messages'],
        tokenize=False,
        add_generation_prompt=True  # 어시스턴트 응답 시작 토큰 추가
    )

    # 프롬프트 토큰화 및 길이 제한
    inputs = tokenizer(
        text=prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 텍스트 생성 파라미터 설정
    generation_config = {
        'max_new_tokens': max_length,        # 최대 생성 토큰 수
        'repetition_penalty': 1.2,           # 반복 방지 페널티
        'no_repeat_ngram_size': 3,           # n-gram 반복 방지
        # 평가를 위한 결정적 생성 설정
        'do_sample': False,                  # 샘플링 비활성화
        'num_beams': 4,                      # 빔 서치 사용
        'pad_token_id': tokenizer.pad_token_id,
        'eos_token_id': tokenizer.eos_token_id,
    }

    # 답변 생성 실행
    with torch.no_grad():
        outputs = model.generate(**inputs, **generation_config)

    # 새로 생성된 토큰만 디코딩
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # 답변 후처리: 중복 문장 제거 및 정리
    response = response.strip()
    sentences = response.split('.')
    unique_sentences = []
    for sent in sentences:
        if sent.strip() and sent.strip() not in unique_sentences:
            unique_sentences.append(sent.strip())
    response = '. '.join(unique_sentences)
    if response and not response.endswith('.'):
        response += '.'

    return response

In [8]:
# ============================================
# Cell 6: 성능 평가 메트릭 및 모델 평가
# ============================================

from collections import Counter
import numpy as np
import random
from tqdm import tqdm
from rouge_score import rouge_scorer

def normalize_answer(text: str) -> str:
    """답변 텍스트 정규화 (대소문자, 공백, 특수문자 처리)"""
    text = text.lower()                                      # 소문자 변환
    text = re.sub(r'\s+', ' ', text)                        # 연속 공백 제거
    text = re.sub(r'[^\w\s가-힣0-9]', '', text)               # 특수문자 제거
    return text.strip()

def compute_f1_score(prediction: str, ground_truth: str) -> float:
    """토큰 레벨 F1 Score 계산"""
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()

    # 빈 예측이나 정답 처리
    if not pred_tokens or not truth_tokens:
        return 0.0

    # 공통 토큰 개수 계산
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    # Precision, Recall, F1 계산
    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)

    return f1

def compute_rouge_scores(prediction: str, ground_truth: str) -> dict:
    """ROUGE-1, ROUGE-2, ROUGE-L 점수 계산"""
    print(f"\n예측 원본: {prediction} / 정규화: {normalize_answer(prediction)}")
    print(f"\n정답 원본: {ground_truth} / 정규화: {normalize_answer(ground_truth)}")

    # ROUGE 스코어러 초기화 (한국어는 stemmer 비활성화)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    scores = scorer.score(normalize_answer(ground_truth), normalize_answer(prediction))

    return {
        'rouge1': scores['rouge1'].fmeasure,   # 1-gram F1
        'rouge2': scores['rouge2'].fmeasure,   # 2-gram F1
        'rougeL': scores['rougeL'].fmeasure    # LCS F1
    }

print("평가 메트릭 정의 완료!")

# 평가용 샘플 선택 (계산 시간 고려)
eval_samples = processed_dataset['validation'].select(range(min(10, len(processed_dataset['validation']))))

# 평가 점수 저장 리스트
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []
f1_scores = []

print("모델 평가 중...")
# 각 샘플에 대해 예측 및 평가 수행
for sample in tqdm(eval_samples):
    print(sample)
    # 모델 예측 생성
    prediction = generate_answer(sample['question'], sample['context'], max_length=256)
    ground_truth = sample['answer']

    # 각종 메트릭 계산
    rouge_scores = compute_rouge_scores(prediction, ground_truth)
    f1 = compute_f1_score(prediction, ground_truth)

    # 점수 리스트에 추가
    rouge1_scores.append(rouge_scores['rouge1'])
    rouge2_scores.append(rouge_scores['rouge2'])
    rougeL_scores.append(rouge_scores['rougeL'])
    f1_scores.append(f1)

# 평균 성능 계산 및 출력
avg_rouge1 = np.mean(rouge1_scores) * 100
avg_rouge2 = np.mean(rouge2_scores) * 100
avg_rougeL = np.mean(rougeL_scores) * 100
avg_f1 = np.mean(f1_scores) * 100

print(f"\n평가 결과:")
print(f"F1 Score: {avg_f1:.2f}%")
print(f"ROUGE-1: {avg_rouge1:.2f}%")
print(f"ROUGE-2: {avg_rouge2:.2f}%")
print(f"ROUGE-L: {avg_rougeL:.2f}%")

# 모델을 평가 모드로 전환
model.eval()

# 실제 예측 결과 샘플 확인
print("\n예측 샘플:")
for i in range(3):
    sample = eval_samples[i]
    prediction = generate_answer(sample['question'], sample['context'], max_length=256)
    print(f"\n질문: {sample['question']}")
    print(f"정답: {sample['answer']}")
    print(f"예측: {prediction}")

# GPU 메모리 정리
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer

torch.cuda.empty_cache()  # GPU 캐시 비우기

print("GPU 메모리 정리 완료!")
print(f"현재 GPU 메모리 사용량: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

평가 메트릭 정의 완료!
모델 평가 중...


  0%|          | 0/10 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


{'instruction': '일본의 일촌일품이란 정책은 무엇인가요?', 'output': '일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다. 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다. 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만, 대형 유원지나 교육시설을 개발하여 이를 주요 명소로 만드는 경우도 있습니다. 이에 따라 일촌일품의 주제는 상당히 폭넓고 다양합니다. 일본의 지리적 특성과 기후 차이를 이용하여 개발하는 경우가 많은데, 이를 흉내내어 우리나라 지자체도 지역경제 활성화를 위해 다양한 축제와 행사를 개최하는 경우가 많습니다. 하지만 일촌일품이 실패하는 경우 지자체는 부채에 시달리는 경우도 있으므로 신중한 기획과 검토가 필요합니다.', 'url': 'https://kin.naver.com/qna/detail.naver?d1id=9&dirId=9020101&docId=74933167', 'id': 'koalpaca_-5297262789721773236', 'context': '', 'question': '일본의 일촌일품이란 정책은 무엇인가요?', 'answer': '일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다. 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다. 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만, 대형 유원지나 교육시설을 개발하여 이를 주요 명소로 만드는 경우도 있습니다. 이에 따라 일촌일품의 주제는 상당히 폭넓고 다양합니다. 일본의 지리적 특성과 기후 차이를 이용하여 개발하는 경우가 많은데, 이를 흉내내어 우리나라 지자체도 지역경제 활성화를 위해 다양한 축제와 행사를 개최하는 경우가 많습니다. 하지만 일촌일품이 실패하는 경우 지자체는 부채에 시달리는 경우도 있으므로 신중한 기획과 검토가 필요합니다.

 10%|█         | 1/10 [00:22<03:25, 22.86s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



예측 원본: 일본은 2019년 4월 1일부터 2022년 3월 31일까지 3년간 일본에서 거주하는 외국인 1,000만 명 이상의 이민자에게 1년간 1인당 100만 원을 지급하는 정책을 시행했습니다. 이 정책은 일본이 2020년 하계 올림픽과 2021년 동계올림픽에 출전할 수 있도록 지원하기 위해 추진되었습니다. 하지만, 이 정책에 대한 비판과 논란이 많았으며, 일본 정부는 이 정책을 중단하기로 결정했습니다. / 정규화: 일본은 2019년 4월 1일부터 2022년 3월 31일까지 3년간 일본에서 거주하는 외국인 1000만 명 이상의 이민자에게 1년간 1인당 100만 원을 지급하는 정책을 시행했습니다 이 정책은 일본이 2020년 하계 올림픽과 2021년 동계올림픽에 출전할 수 있도록 지원하기 위해 추진되었습니다 하지만 이 정책에 대한 비판과 논란이 많았으며 일본 정부는 이 정책을 중단하기로 결정했습니다

정답 원본: 일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다. 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다. 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만, 대형 유원지나 교육시설을 개발하여 이를 주요 명소로 만드는 경우도 있습니다. 이에 따라 일촌일품의 주제는 상당히 폭넓고 다양합니다. 일본의 지리적 특성과 기후 차이를 이용하여 개발하는 경우가 많은데, 이를 흉내내어 우리나라 지자체도 지역경제 활성화를 위해 다양한 축제와 행사를 개최하는 경우가 많습니다. 하지만 일촌일품이 실패하는 경우 지자체는 부채에 시달리는 경우도 있으므로 신중한 기획과 검토가 필요합니다. / 정규화: 일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만 대형 유원

...


예측 원본: 어두울 때 잠이 오는 이유는 신체 내부의 호르몬 수치가 낮아지기 때문입니다. 이때 호르โมन 수치의 감소로 인해 신경계가 자극을 받지 않아 잠에 빠지게 됩니다. 또한, 신체의 온도가 낮아지는 것도 잠을 자는 이유 중 하나입니다. 신체는 온도를 조절하기 위해 호흡을 조절하며, 온도 감소와 동시에 호호를 증가시킵니다. 이러한 호호는 신경을 자극하여 잠을 일으키는 역할을 합니다. 새벽과 저녁에는 기압의 차이가 크게 나타나는데, 기압 차이가 커질수록 호호가 증가하여 신경이 자극되어 잠이 자는 경향이 높아집니다. / 정규화: 어두울 때 잠이 오는 이유는 신체 내부의 호르몬 수치가 낮아지기 때문입니다 이때 호르โมन 수치의 감소로 인해 신경계가 자극을 받지 않아 잠에 빠지게 됩니다 또한 신체의 온도가 낮아지는 것도 잠을 자는 이유 중 하나입니다 신체는 온도를 조절하기 위해 호흡을 조절하며 온도 감소와 동시에 호호를 증가시킵니다 이러한 호호는 신경을 자극하여 잠을 일으키는 역할을 합니다 새벽과 저녁에는 기압의 차이가 크게 나타나는데 기압 차이가 커질수록 호호가 증가하여 신경이 자극되어 잠이 자는 경향이 높아집니다

정답 원본: 어두운 환경에서는 멜라토닌 호르몬 분비량이 증가하여 잠을 자게 됩니다. 멜라토닌은 생체리듬을 주관하는 작용을 지니며, 망막에 도달하는 빛의 양에 따라 분비량이 조절됩니다. 또한, 멜라토닌 제가 수면 장애와 시차로 인한 피로회복에 탁월한 효과를 발휘하고 있습니다. 기압과 수면과의 관련성에 대해서는 더 많은 연구가 필요합니다. / 정규화: 어두운 환경에서는 멜라토닌 호르몬 분비량이 증가하여 잠을 자게 됩니다 멜라토닌은 생체리듬을 주관하는 작용을 지니며 망막에 도달하는 빛의 양에 따라 분비량이 조절됩니다 또한 멜라토닌 제가 수면 장애와 시차로 인한 피로회복에 탁월한 효과를 발휘하고 있습니다 기압과 수면과의 관련성에 대해서는 더 많은 연구가 필요합니다

평가 결과:
F1 Score: 11.30%
ROUGE-1: 5.92%
ROUGE-2: 0.

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



질문: 일본의 일촌일품이란 정책은 무엇인가요?
정답: 일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다. 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다. 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만, 대형 유원지나 교육시설을 개발하여 이를 주요 명소로 만드는 경우도 있습니다. 이에 따라 일촌일품의 주제는 상당히 폭넓고 다양합니다. 일본의 지리적 특성과 기후 차이를 이용하여 개발하는 경우가 많은데, 이를 흉내내어 우리나라 지자체도 지역경제 활성화를 위해 다양한 축제와 행사를 개최하는 경우가 많습니다. 하지만 일촌일품이 실패하는 경우 지자체는 부채에 시달리는 경우도 있으므로 신중한 기획과 검토가 필요합니다.
예측: 일본은 2019년 4월 1일부터 2022년 3월 31일까지 3년간 일본에서 거주하는 외국인 1,000만 명 이상의 이민자에게 1년 1회에 한 번씩 100만 원을 지불하도록 하는 제도입니다. 이 제도의 목적은 외국인들이 일본에 정착할 수 있도록 유도하기 위해서입니다. 또한, 이 제도는 일본의 인구 감소 문제를 해결하고, 외국인들의 경제적 활동을 촉진하기 위해서도 사용됩니다.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



질문: 냉장고, 전기밥솥의 에너지소비효율등급 기준이 강화된다고 하던데, 사실인가요?
정답: 네, 맞습니다. 산업통상자원부에서는 1 2 등급 비중이 과도해 지는 냉장고, 전기밥솥, 공기청정기, 냉온수기 등 4개 제품의 에너지소비효율등급 기준을 상향 조정하였습니다. 전기냉장고와 전기밥솥은 각각 1등급 기준을 현행 대비 20%, 15% 상향 조정했으며, 공기청정기는 2등급 기준을 현행 대비 30% 상향 조정하였습니다. 또한, 전기냉온수기도 1등급 기준을 현행 대비 20% 상향 조정하였고, 적용 범위도 빙축열 방식을 추가하여 보급이 확대됩니다. 이를 통해 연간 118GWh의 전력사용량 절감 약 189억원 과 5만 톤의 온실가스 CO2 저감 효과가 있을 것으로 예상됩니다. 이에 따라, 제품의 효율 향상으로 인해 에너지효율 기준이 강화됩니다.
예측: 현재 우리나라에서 사용되고 있는 냉장보일러의 전기 소비 효율 등급은 2006년 1월 1일부터 2007년 12월 31일까지는 1 2 3 등급으로, 2008년 2월 29일부터는 4 5 6 등급이 사용되고 있습니다. 2009년 6월 30일부터에는 7 8 9 등급까지도 사용되고 있으며, 2010년 7월 25일에는 10 11 12 등급도 사용되었습니다. 현재는 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72.

질문: 공포영화를 관람 중 심장마비로 사망한다면 누구의 책임일까요? 감독에게도 책임을 물을 수 있을까요?
정답: 만약 공포영화를 관람하는 관객이 자신의 심장이 약하다는 것을 알고 있었다면, 관객 본인의 책임입니다. 그러나 극장 운영자에게는 보호의무가 있으므로, 임산부, 어린이, 노약자, 심장약한 분들에게는 관람을 삼가하라는 안내문구를 붙여야 합니다. 만약 